# Задача 1. Значение фильтра → value_id

LLM сформировала `'утюг' FILTER { color=красный }`. Приложению нужен ID значения из справочника **этого ключа**.

**Контракт результата**
- `color=краснй` → `['c_red']`.
- `color=селый` → `['c_white', 'c_gray']` — два значения на одинаковом расстоянии.
- `material=красный` → `[]` — поиск ограничен материалами.

Верните все ID с минимальным расстоянием, если оно не превышает 1. При точном совпадении верните только точные совпадения. Пустой ввод, неизвестный ключ и отсутствие подходящего значения дают `[]`. Допуск в одну вставку, удаление или замену — учебная настройка.

Справочник и ID учебные. Код парсинга уже дан для запросов с одним фильтром.

**Версия с решением.** Все ячейки выполнены, результаты сохранены.

In [1]:
import re
import unicodedata
from rapidfuzz.distance import Levenshtein

catalog = {
    "color": [("c_red", "красный"), ("c_white", "белый"), ("c_gray", "серый")],
    "material": [("m_steel", "сталь"), ("m_cotton", "хлопок")],
    "size": [("s_small", "маленький"), ("s_large", "большой")],
}

def normalize(text):
    text = unicodedata.normalize("NFKC", text).casefold().replace("ё", "е")
    return re.sub(r"[\W_]+", " ", text).strip()

## Шаг 1. Получаем ключ и значение

Парсер выделяет строку запроса, ключ фильтра и значение. Нормализация объединяет регистр, пробелы и пунктуацию. Все сравнения выполняются после одинаковой обработки обеих строк.

In [2]:
def parse_query(text):
    pattern = r"'(?P<query>[^']*)'\s+FILTER\s*\{\s*(?P<key>\w+)\s*=\s*(?P<value>[^{}]+?)\s*\}"
    match = re.fullmatch(pattern, text.strip())
    if match is None:
        raise ValueError("Ожидается запрос с одним FILTER { key=value }")
    return match.groupdict()

parsed = parse_query("'утюг' FILTER { color=краснй }")
print(parsed)
print(normalize("  КРАСНЫЙ! "))

{'query': 'утюг', 'key': 'color', 'value': 'краснй'}
красный


## Шаг 2. Сравниваем со значениями нужного ключа · TODO 1

Заполните две строки. Получите список из `catalog` через `.get(key, [])`. Для каждой пары `(value_id, value)` посчитайте `Levenshtein.distance` между нормализованными строками.

Функция возвращает пары `(ID, расстояние)`. Расстояние 0 означает совпадение, 1 — одну правку.

In [3]:
def value_distances(key, text):
    values = catalog.get(key, [])
    query = normalize(text)
    return [(value_id, Levenshtein.distance(query, normalize(value)))
            for value_id, value in values]

assert dict(value_distances("color", "краснй"))["c_red"] == 1
assert value_distances("unknown", "красный") == []
assert {value_id for value_id, _ in value_distances("material", "красный")} == {"m_steel", "m_cotton"}
print(value_distances("color", "селый"))

[('c_red', 5), ('c_white', 1), ('c_gray', 1)]


## Шаг 3. Возвращаем все лучшие совпадения · TODO 2

Найдите минимальное расстояние. Если оно больше `max_distance`, верните пустой список. Иначе верните **все** ID с этим расстоянием, сохранив порядок справочника. Обработка пустых входов уже дана.

In [4]:
def match_value(key, text, max_distance=1):
    pairs = value_distances(key, text)
    if not normalize(text) or not pairs:
        return []
    best_distance = min(distance for _, distance in pairs)
    if best_distance > max_distance:
        return []
    return [value_id for value_id, distance in pairs if distance == best_distance]

assert match_value("color", "краснй") == ["c_red"]
assert set(match_value("color", "селый")) == {"c_white", "c_gray"}
assert match_value("color", "белый") == ["c_white"]
assert match_value("material", "красный") == []
assert match_value("color", "") == []
assert match_value("unknown", "красный") == []
assert match_value("color", "краснй", max_distance=0) == []

## Шаг 4. Подключаем к запросу LLM

Исходная строка поиска сохраняется. Список `value_ids` задаёт допустимые значения фильтра. Для пустого списка вызывающий код запрашивает уточнение. Для нескольких ID он может передать их в индекс как OR по значениям одного ключа.

In [5]:
def resolve_query(text):
    parsed = parse_query(text)
    ids = match_value(parsed["key"], parsed["value"])
    return {"query": parsed["query"], "filter": {"key": parsed["key"], "value_ids": ids}}

for value in ["красный", "краснй", "крассный", "селый", "фиолетовый"]:
    print(resolve_query(f"'утюг' FILTER {{ color={value} }}"))

{'query': 'утюг', 'filter': {'key': 'color', 'value_ids': ['c_red']}}
{'query': 'утюг', 'filter': {'key': 'color', 'value_ids': ['c_red']}}
{'query': 'утюг', 'filter': {'key': 'color', 'value_ids': ['c_red']}}
{'query': 'утюг', 'filter': {'key': 'color', 'value_ids': ['c_white', 'c_gray']}}
{'query': 'утюг', 'filter': {'key': 'color', 'value_ids': []}}


## Шаг 5. Проверяем на размеченных примерах

Справочник конечен. Например, настоящий цвет `сизый` в нём отсутствует, поэтому правильный ответ для него — пустой список. Для `селый` разметка содержит оба одинаково подходящих значения.

Выполните готовый код. Таблица покажет пропуски и лишние ID для каждого запроса. Дополните выборку своим примером.

In [6]:
import pandas as pd

# Ключ, значение LLM, множество всех допустимых ID.
cases = [
    ("color", "красный", {"c_red"}),
    ("color", "КРАСНЫЙ!", {"c_red"}),
    ("color", "краснй", {"c_red"}),
    ("color", "крассный", {"c_red"}),
    ("color", "крснй", {"c_red"}),
    ("color", "селый", {"c_white", "c_gray"}),
    ("material", "стал", {"m_steel"}),
    ("material", "красный", set()),
    ("color", "сизый", set()),
    ("color", "", set()),
]

In [7]:
def evaluate_filters(max_distance):
    rows = []
    for key, value, gold in cases:
        predicted = set(match_value(key, value, max_distance))
        rows.append({"key": key, "value": value, "gold": sorted(gold),
                     "predicted": sorted(predicted), "TP": len(gold & predicted),
                     "FP": len(predicted - gold), "FN": len(gold - predicted),
                     "correct": predicted == gold})
    return pd.DataFrame(rows)

display(evaluate_filters(max_distance=1))

,key,value,gold,predicted,TP,FP,FN,correct
0,color,красный,[c_red],[c_red],1,0,0,True
1,color,КРАСНЫЙ!,[c_red],[c_red],1,0,0,True
2,color,краснй,[c_red],[c_red],1,0,0,True
3,color,крассный,[c_red],[c_red],1,0,0,True
4,color,крснй,[c_red],[],0,0,1,False
5,color,селый,"[c_gray, c_white]","[c_gray, c_white]",2,0,0,True
6,material,стал,[m_steel],[m_steel],1,0,0,True
7,material,красный,[],[],0,0,0,True
8,color,сизый,[],[],0,0,0,True
9,color,,[],[],0,0,0,True


## Шаг 6. Считаем метрики и меняем порог · TODO 3

Суммируем совпавшие ID, лишние ID и пропуски по всем запросам. Это micro-усреднение.

$$Precision=\frac{TP}{TP+FP},\qquad Recall=\frac{TP}{TP+FN}.$$

**Полностью правильный ответ** означает точное совпадение множеств ID. Для пустой разметки правильным будет только пустой ответ. Такой запрос влияет на Precision при ложном выборе и на долю правильных ответов. При отсутствии предсказаний в таблице условно ставим Precision равным 0.

Реализуйте `metric_values` по двум формулам выше. Для доли полностью правильных ответов возьмите среднее списка `correct` из True/False. Если знаменатель равен нулю, верните 0.0. Готовая функция ниже подставит подсчитанные TP, FP, FN.

Предскажите результат для 0, 1 и 2 правок, затем запустите код. Контракт задачи использует одну правку. Другие пороги исследуем для понимания ошибок.

In [8]:
def metric_values(tp, fp, fn, correct):
    return {"Precision": tp / (tp + fp) if tp + fp else 0.0,
            "Recall": tp / (tp + fn) if tp + fn else 0.0,
            "Полностью верных": sum(correct) / len(correct) if len(correct) else 0.0}

assert metric_values(3, 1, 2, [True, False]) == {
    "Precision": 0.75, "Recall": 0.6, "Полностью верных": 0.5}
assert metric_values(0, 0, 1, [False])["Precision"] == 0.0
assert metric_values(0, 1, 0, [False])["Recall"] == 0.0
assert metric_values(0, 0, 0, [])["Полностью верных"] == 0.0

In [9]:
def summarize_filters(max_distance):
    details = evaluate_filters(max_distance)
    tp, fp, fn = details[["TP", "FP", "FN"]].sum()
    return {"max_distance": max_distance,
            **metric_values(tp, fp, fn, details["correct"]),
            "Лишних ID": fp, "Пропущенных ID": fn}

display(pd.DataFrame([summarize_filters(d) for d in [0, 1, 2]]).round(3))

,max_distance,Precision,Recall,Полностью верных,Лишних ID,Пропущенных ID
0,0,1.000,0.250,0.5,0,6
1,1,1.000,0.875,0.9,0,1
2,2,0.889,1.000,0.9,1,0


**Разберите два запроса.** При каком пороге восстанавливается `крснй`? Когда `сизый` ошибочно превращается в известный цвет? Почему высокая полнота сама по себе не гарантирует корректный фильтр?

Эта выборка предназначена для разбора поведения. Для выбора рабочего порога потребуется отдельная разметка реальных ошибок LLM и итоговая проверка на отложенных запросах.

**Проверка понимания.** Почему для `белый` следует вернуть один ID, хотя `серый` тоже попадает в достаточно большой радиус? Что произойдёт, если искать сразу по всем ключам?

Вторая задача использует эти же нормализацию и редакционное расстояние в готовом коде. Новая работа начинается с поиска границ упоминания.